# Lost in the Museum - DINOv2 ViT-g/14 @ 518px

Scales the winning recipe (ViT-L @518, CLS+GeM, PCA-1536 whitened, LB 0.79194)
up to **ViT-g/14** - 1.1B parameters against ViT-L's 300M.

This was attempted on a 16 GB Apple-silicon laptop and failed twice: the model
plus 518px activations exceeded unified memory and the machine swap-thrashed
down to 0.2 img/s. A Kaggle P100/T4 has dedicated VRAM, so it fits here.

**Settings:** GPU accelerator ON, Internet ON (weights come from `torch.hub`).
Expect roughly 60-90 minutes.

Half precision is used for the weights: it halves VRAM and cosine similarity is
robust to the small loss of precision.


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset

from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None

WORK = Path("/kaggle/working")

def find_data_dir():
    """Locate the image folder without hard-coding a slug.

    The mounted path differs depending on whether the data is attached as the
    competition or as a standalone dataset, so we just find the directory that
    actually holds the PNGs.
    """
    best, best_n = None, 0
    for d in Path("/kaggle/input").rglob("*"):
        if not d.is_dir():
            continue
        n = sum(1 for _ in d.glob("*.png"))
        if n > best_n:
            best, best_n = d, n
    return best, best_n

DATA_DIR, _n = find_data_dir()
print("Data:", DATA_DIR, f"({_n} png)")
assert DATA_DIR is not None, "No .png folder found under /kaggle/input -- attach the competition data"

MODEL, SIZE, DIM, BATCH, FP16 = "dinov2_vitg14", 518, 1536, 4, True

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", (
    "No GPU. Settings -> Accelerator -> GPU T4 x2 (or P100), then re-run. "
    "ViT-g on CPU would take days.")
paths = sorted(DATA_DIR.glob("*.png"))
print(f"device={device}  images={len(paths)}  model={MODEL}@{SIZE}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")
assert len(paths) == 20000, f"expected 20000 images, found {len(paths)}"

## Preprocessing and pooling

Images are resized directly to `(SIZE, SIZE)`. An aspect-preserving letterbox
was tested and scored *worse* on the leaderboard (0.765 vs 0.775 at 392px), so
the plain resize is kept.

Each image is described by the CLS token concatenated with GeM-pooled patch
tokens, each L2-normalised first so neither half dominates. GeM (generalised
mean, p=3) weights the most distinctive regions and survives cropping better
than CLS alone.


In [ ]:
class ImageFolder(Dataset):
    def __init__(self, paths, size):
        self.paths, self.size = paths, size
        self.tf = transforms.Compose([
            transforms.Resize((size, size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.paths[i]).convert("RGB")), i
        except Exception as e:
            print(f"  ! failed {self.paths[i].name}: {e}")
            return torch.zeros(3, self.size, self.size), i   # never drop a row


def gem_pool(patch_tokens, p=3.0, eps=1e-6):
    return patch_tokens.clamp(min=eps).pow(p).mean(dim=1).pow(1.0 / p)

## Extraction, with checkpointing

Progress is written to `/kaggle/working` every few minutes and reloaded on
start. If the session dies partway, re-running this notebook resumes instead of
restarting from zero.


In [ ]:
model = torch.hub.load("facebookresearch/dinov2", MODEL, verbose=False).eval().to(device)
if FP16:
    model = model.half()

CKPT_F, CKPT_D = WORK / "ckpt_feats.npy", WORK / "ckpt_done.npy"
feats, done_mask = None, np.zeros(len(paths), dtype=bool)
if CKPT_F.exists() and CKPT_D.exists():
    feats, done_mask = np.load(CKPT_F), np.load(CKPT_D)
    print(f"Resuming: {done_mask.sum()}/{len(paths)} already done")

todo = np.flatnonzero(~done_mask)
loader = DataLoader(Subset(ImageFolder(paths, SIZE), todo.tolist()),
                    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

def save_ckpt():
    np.save(str(CKPT_F) + ".tmp.npy", feats)
    np.save(str(CKPT_D) + ".tmp.npy", done_mask)
    Path(str(CKPT_F) + ".tmp.npy").replace(CKPT_F)
    Path(str(CKPT_D) + ".tmp.npy").replace(CKPT_D)

t0 = last = time.time()
done = start_done = int(done_mask.sum())
with torch.no_grad():
    for batch, idxs in loader:
        batch = batch.to(device, non_blocking=True)
        if FP16:
            batch = batch.half()
        out = model.forward_features(batch)
        vec = torch.cat([
            F.normalize(out["x_norm_clstoken"], dim=1),
            F.normalize(gem_pool(out["x_norm_patchtokens"]), dim=1),
        ], dim=1).float().cpu().numpy()

        if feats is None:
            feats = np.zeros((len(paths), vec.shape[1]), dtype=np.float32)
        feats[idxs.numpy()] = vec
        done_mask[idxs.numpy()] = True
        done += len(idxs)

        if time.time() - last > 300:
            save_ckpt(); last = time.time()
        if done % (BATCH * 100) < BATCH:
            r = max(done - start_done, 1) / (time.time() - t0)
            print(f"  {done}/{len(paths)}  {r:.1f} img/s  ETA {(len(paths)-done)/r/60:.0f} min", flush=True)

assert done_mask.all(), f"only {done_mask.sum()}/{len(paths)} extracted"
print(f"Extracted {feats.shape} in {(time.time()-t0)/60:.1f} min")

# Persist the raw (un-reduced) features so the PCA dimension can be tuned
# offline without paying for another GPU pass. ~245 MB, well inside the
# 19.5 GB working-directory quota.
np.save("/kaggle/working/features_g.npy", feats)
np.save("/kaggle/working/feature_names.npy", np.array([p.name for p in paths]))
print("saved raw features for offline dimension sweep")

## Whitened PCA, then submission

Order is: L2-normalise, PCA **with whitening**, L2-normalise again.

Whitening was the single most valuable ingredient in this competition -
whitened 512-d beat un-whitened 2048-d. The gallery holds 9,000 artwork
distractors, so the dominant variance directions all encode "this is a
painting", shared by the true match and the distractors alike. Whitening
equalises the component scales so discriminative detail is not drowned out.


In [ ]:
def l2(x, eps=1e-12):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)

x = l2(feats.astype(np.float64))
mu = x.mean(axis=0, keepdims=True)
_, s, vt = np.linalg.svd(x - mu, full_matrices=False)      # exact, deterministic
dim = min(DIM, x.shape[1])
x = l2((x - mu) @ vt[:dim].T / (s[:dim] / np.sqrt(len(x) - 1) + 1e-8)).astype(np.float32)
print(f"PCA {feats.shape[1]} -> {dim}   explains {(s[:dim]**2).sum()/(s**2).sum():.1%} of variance")

df = pd.DataFrame(x, columns=[f"feature_{i}" for i in range(x.shape[1])])
df.insert(0, "image_name", [p.name for p in paths])
df["ID"] = df["image_name"]
df.to_csv(WORK / "submission.csv", index=False, float_format="%.6f")

for f in (CKPT_F, CKPT_D):
    f.unlink(missing_ok=True)   # completed; free the working-dir quota

print(f"rows={len(df)}  cols={df.shape[1]}  norms~{np.linalg.norm(x,axis=1).mean():.4f}")
df.iloc[:3, :5]